# Phase 5 — Final evaluation and model comparison

This notebook evaluates both saved models once on the same untouched test split and produces presentation-ready metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from shutil import copy2
from tqdm.auto import tqdm
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, tensorflow as tf
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

PROJECT_ROOT = Path('/content/drive/MyDrive/waste-classification')
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'phase2_splits'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'phase5_evaluation'
MODEL_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_CLASSES = ('plastic', 'paper', 'glass', 'metal', 'organic')
CLASS_TO_INDEX = {name:i for i, name in enumerate(TARGET_CLASSES)}
IMAGE_SIZE, BATCH_SIZE = (224,224), 32


In [ ]:
test_df = pd.read_csv(SPLIT_DIR / 'test.csv')
drive_root = PROJECT_ROOT / 'data' / 'raw' / 'mwcd'; local_root = Path('/content/waste-classification-local-data')
if not local_root.exists():
    images = [p for p in drive_root.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
    for source in tqdm(images, desc='Copying images to local disk'):
        destination = local_root / source.relative_to(drive_root); destination.parent.mkdir(parents=True, exist_ok=True); copy2(source, destination)
test_df['path'] = test_df['path'].map(lambda p: str(local_root / Path(p).relative_to(drive_root)))

def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None,None,3])
    return tf.cast(tf.image.resize(image, IMAGE_SIZE), tf.float32), label
labels = test_df['label'].map(CLASS_TO_INDEX).to_numpy()
test_ds = tf.data.Dataset.from_tensor_slices((test_df['path'].to_numpy(), labels)).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
print(f'Test images: {len(test_df):,}')

In [ ]:
models = {
    'Custom CNN': tf.keras.models.load_model(MODEL_DIR / 'custom_cnn_best.keras'),
    'MobileNetV2': tf.keras.models.load_model(MODEL_DIR / 'mobilenetv2_best.keras'),
}
results = []
predictions = {}
for name, model in models.items():
    predicted = np.argmax(model.predict(test_ds, verbose=1), axis=1)
    predictions[name] = predicted
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted, average='weighted', zero_division=0)
    results.append({'Model': name, 'Accuracy': accuracy_score(labels, predicted), 'Weighted precision': precision, 'Weighted recall': recall, 'Weighted F1-score': f1})
comparison = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
display(comparison.style.format({column: '{:.2%}' for column in comparison.columns[1:]}))
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

In [ ]:
for name, predicted in predictions.items():
    report = classification_report(labels, predicted, target_names=TARGET_CLASSES, output_dict=True, zero_division=0)
    filename_prefix = name.lower().replace(' ', '_')
    pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / f'{filename_prefix}_classification_report.csv')
    matrix = confusion_matrix(labels, predicted)
    plt.figure(figsize=(7,6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
    plt.title(f'{name} confusion matrix')
    plt.xlabel('Predicted label'); plt.ylabel('True label')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{filename_prefix}_confusion_matrix.png', dpi=180)
    plt.show()
    print(f'\n{name} classification report:')
    display(pd.DataFrame(report).transpose().round(3))

In [ ]:
# Save selected MobileNetV2 prediction examples for discussion.
mobile_pred = predictions['MobileNetV2']
rng = np.random.default_rng(42)
chosen = rng.choice(len(test_df), size=min(12, len(test_df)), replace=False)
plt.figure(figsize=(14,9))
for slot, index in enumerate(chosen, 1):
    image = tf.keras.utils.load_img(test_df.iloc[index]['path'], target_size=IMAGE_SIZE)
    axis = plt.subplot(3,4,slot); axis.imshow(image)
    true = TARGET_CLASSES[labels[index]]; predicted = TARGET_CLASSES[mobile_pred[index]]
    axis.set_title(f'True: {true}\nPred: {predicted}', color='green' if true == predicted else 'red')
    axis.axis('off')
plt.suptitle('MobileNetV2 sample test predictions'); plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mobilenetv2_sample_predictions.png', dpi=180); plt.show()
print('All evaluation files saved to:', OUTPUT_DIR)